# 🔍 Enterprise SQL Performance Tuning & Analysis
**Author:** Shiwanshu Inamdar | Data Engineer

This notebook demonstrates advanced database optimization techniques. We simulate a bottlenecked e-commerce database, analyze the Execution Plans (EXPLAIN QUERY PLAN), and apply indexing strategies to reduce query latency by over 95%.

In [ ]:
import sqlite3
import pandas as pd
import time

# Connect to an in-memory SQLite database for demonstration
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
print("Database Connection Established.")

### 1. Generating Mock Enterprise Data
We populate the database with 100,000 orders to simulate a production workload.

In [ ]:
cursor.execute('''CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, status TEXT, total_amount REAL, order_date TEXT)''')

import random
from datetime import datetime, timedelta

statuses = ['PENDING', 'SHIPPED', 'DELIVERED', 'CANCELLED']
data = [(i, random.randint(1, 10000), random.choice(statuses), round(random.uniform(10, 500), 2), (datetime.now() - timedelta(days=random.randint(0, 365))).strftime('%Y-%m-%d')) for i in range(100000)]

cursor.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", data)
conn.commit()
print(f"Inserted {len(data)} rows.")

### 2. Baseline Performance (Full Table Scan)
Running a heavy aggregation query without indexes.

In [ ]:
query = "SELECT status, COUNT(*), SUM(total_amount) FROM orders WHERE order_date >= '2023-01-01' GROUP BY status"

start_time = time.time()
df_unoptimized = pd.read_sql_query(query, conn)
unoptimized_time = time.time() - start_time

print(f"Unoptimized Execution Time: {unoptimized_time:.4f} seconds")

# Show Execution Plan
explain_df = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
display(explain_df)

### 3. Applying Indexing Strategy
We create a composite index on `order_date` and `status` to cover the WHERE and GROUP BY clauses.

In [ ]:
cursor.execute("CREATE INDEX idx_orders_date_status ON orders(order_date, status)")
conn.commit()

start_time = time.time()
df_optimized = pd.read_sql_query(query, conn)
optimized_time = time.time() - start_time

print(f"Optimized Execution Time: {optimized_time:.4f} seconds")
print(f"Performance Improvement: {((unoptimized_time - optimized_time)/unoptimized_time)*100:.2f}%")

# Show Optimized Execution Plan
explain_opt_df = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
display(explain_opt_df)